- run ADLS access setup

In [0]:
from common_scripts.common_functions import * 

setup_adls_access(spark, dbutils)


In [0]:
import requests
import json
import time
from datetime import datetime, date

i = 0
while i<21:
# getting the data from CoinGecko API 
    url = "https://api.coingecko.com/api/v3/coins/markets"

    params = {
        "vs_currency": "usd",
        "order": "market_cap_desc",
        "per_page": 10,
        "page": i
    }

    response = requests.get(url, params=params)



    if response.status_code != 200:
            if response.status_code == 429:
                print("Rate limit hit. Sleeping...")
                time.sleep(60)
                continue
            else:
                raise Exception(f"API failed with status {response.status_code}")


    data = response.json()

    # Adding ingetion time to each record 
    ingestion_time = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")

    for record in data:
        record["ingestion_time"] = ingestion_time

    # Writing the data into ADLS bronze layer
    today = date.today().strftime("%Y-%m-%d")
    base_path = f"abfss://financial-data-project@financialdatalakestorage.dfs.core.windows.net/bronze/crypto/ingestion_date={today}"
    file_path = f"{base_path}/raw_data_{i}.json"
    dbutils.fs.put(file_path, json.dumps(data), overwrite=True)

    print(f"Data written to: {file_path}")
    i += 1